# Lab 2 alternative — Fine-tune Parakeet with the NGC NeMo Speech container

This notebook reproduces Lab 2's bounded Dutch FLEURS experiment while moving model loading, baseline evaluation, fine-tuning, checkpoint selection, final WER/CER evaluation, TensorBoard logging, and `.nemo` export into an NVIDIA NGC NeMo container. The host notebook only prepares the audio/manifests and launches the container.

This is a **NeMo framework container**, not a fine-tuning microservice API. A container removes most host-side CUDA/PyTorch/NeMo dependency setup, but it does not remove the training recipe, manifests, GPU sizing, checkpoints, evaluation, or artifact handling. The default is the official consolidated `nvcr.io/nvidia/nemo:24.12` image because its CUDA 12.6 generation is compatible with the workshop host's 565 driver. The newer `nemo-speech:26.07.00` image uses CUDA 13.2 and requires driver 595.58 or later. See the [NeMo 24.12 container instructions](https://docs.nvidia.com/nemo-framework/user-guide/24.12/installation.html), [current NGC NeMo Speech container](https://catalog.ngc.nvidia.com/orgs/nvidia/-/containers/nemo-speech/26.07.00), and [NeMo ASR fine-tuning guide](https://docs.nvidia.com/nemo/speech/nightly/asr/fine_tuning.html).


In [ ]:
from getpass import getpass
from pathlib import Path
from tempfile import TemporaryDirectory
import json, os, shutil, subprocess, sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'voice_asr_lab').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'voice_asr_lab').exists():
    raise RuntimeError('Open this notebook from the workshop repository.')
sys.path.insert(0, str(ROOT / 'src'))
os.environ.setdefault('HF_HOME', str(ROOT / '.cache' / 'huggingface'))
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

from voice_asr_lab.audio import duration_seconds, load_dummy_librispeech, load_fleurs_records, total_duration
from voice_asr_lab.nemo import write_nemo_manifest
from voice_asr_lab.profiles import (
    COMMON_EFFECTIVE_BATCH_SIZE, COMMON_EVAL_BATCH_SIZE, COMMON_MAX_AUDIO_SECONDS,
    COMMON_TEST_EXAMPLES, COMMON_TRAIN_EXAMPLES, COMMON_VALIDATION_EVERY_EXAMPLES,
    COMMON_VALIDATION_EXAMPLES, detect_profile,
)

if shutil.which('docker') is None:
    raise RuntimeError('Docker is required for the containerized Lab 2 alternative.')
subprocess.run(['docker', 'version', '--format', '{{.Server.Version}}'], check=True)
gpu_inventory = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    text=True,
).strip()
print(gpu_inventory)
HOST_DRIVER_VERSION = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'], text=True,
).splitlines()[0].strip()


## 1. Choose the same experiment controls as Lab 2

The container path deliberately keeps the same model, official splits, example limits, effective batch size, optimizer steps, learning rate, trainable encoder tail, validation cadence, seed, WER checkpoint selection, CER reporting, and English guardrail. It writes separate container-specific checkpoints and outputs so it cannot overwrite the standard Lab 2 result.


In [ ]:
profile = detect_profile()

NEMO_CONTAINER_IMAGE = 'nvcr.io/nvidia/nemo:24.12'
LATEST_NEMO_SPEECH_IMAGE = 'nvcr.io/nvidia/nemo-speech:26.07.00'  # Requires driver >= 595.58.
MINIMUM_DRIVER_BY_IMAGE = {
    'nvcr.io/nvidia/nemo:24.12': '560.35.05',
    'nvcr.io/nvidia/nemo-speech:26.07.00': '595.58',
}
PULL_IMAGE = True  # Set False only after this exact image is cached locally.
MODEL_ID = 'nvidia/parakeet-ctc-0.6b'
LANGUAGE_CONFIG = 'nl_nl'
LANGUAGE_NAME = 'Dutch'
TRAIN_EXAMPLES = COMMON_TRAIN_EXAMPLES
VALIDATION_EXAMPLES = COMMON_VALIDATION_EXAMPLES
TEST_EXAMPLES = COMMON_TEST_EXAMPLES
ENGLISH_GUARDRAIL_EXAMPLES = 4
MAX_AUDIO_SECONDS = COMMON_MAX_AUDIO_SECONDS
MAX_STEPS = profile.train_steps
LEARNING_RATE = profile.learning_rate
TRAINABLE_ENCODER_LAYERS = profile.trainable_encoder_layers
TRAIN_BATCH_SIZE = profile.train_batch_size
EVAL_BATCH_SIZE = COMMON_EVAL_BATCH_SIZE
ACCUMULATE_GRAD_BATCHES = profile.gradient_accumulation_steps
VALIDATION_EVERY_EXAMPLES = COMMON_VALIDATION_EVERY_EXAMPLES
RANDOM_SEED = 7
ENABLE_TENSORBOARD = True

def driver_tuple(value):
    return tuple(int(part) for part in value.split('.'))

required_driver = MINIMUM_DRIVER_BY_IMAGE.get(NEMO_CONTAINER_IMAGE)
if required_driver and driver_tuple(HOST_DRIVER_VERSION) < driver_tuple(required_driver):
    raise RuntimeError(
        f'{NEMO_CONTAINER_IMAGE} requires NVIDIA driver {required_driver} or later; '
        f'this host has {HOST_DRIVER_VERSION}. Select a compatible image or upgrade the host driver.'
    )
assert TRAIN_BATCH_SIZE * ACCUMULATE_GRAD_BATCHES == COMMON_EFFECTIVE_BATCH_SIZE
assert VALIDATION_EVERY_EXAMPLES % TRAIN_BATCH_SIZE == 0
print({
    'container_image': NEMO_CONTAINER_IMAGE, 'host_driver': HOST_DRIVER_VERSION,
    'minimum_driver': required_driver, 'profile': profile.name,
    'model': MODEL_ID, 'language': LANGUAGE_CONFIG,
    'requested_examples': (TRAIN_EXAMPLES, VALIDATION_EXAMPLES, TEST_EXAMPLES),
    'max_audio_seconds': MAX_AUDIO_SECONDS, 'max_optimizer_steps': MAX_STEPS,
    'learning_rate': LEARNING_RATE,
    'trainable_encoder_layers': TRAINABLE_ENCODER_LAYERS,
    'effective_batch_size': TRAIN_BATCH_SIZE * ACCUMULATE_GRAD_BATCHES,
    'tensorboard_enabled': ENABLE_TENSORBOARD,
})


## 2. Prepare the identical mounted NeMo manifests

The host kernel already contains the dataset/audio tooling used by Lab 2. It writes lossless 16 kHz WAV files and absolute manifest paths beneath the repository. Docker mounts the repository at the same absolute path, so the container sees every manifest and audio file without copying or rewriting paths.


In [ ]:
train_records = load_fleurs_records(
    LANGUAGE_CONFIG, 'train', limit=TRAIN_EXAMPLES, max_audio_seconds=MAX_AUDIO_SECONDS
)
validation_records = load_fleurs_records(
    LANGUAGE_CONFIG, 'validation', limit=VALIDATION_EXAMPLES, max_audio_seconds=MAX_AUDIO_SECONDS
)
test_records = load_fleurs_records(
    LANGUAGE_CONFIG, 'test', limit=TEST_EXAMPLES, max_audio_seconds=MAX_AUDIO_SECONDS
)
english_records = [
    row for row in load_dummy_librispeech(limit=30)
    if duration_seconds(row['audio'], row['sampling_rate']) <= MAX_AUDIO_SECONDS
][:ENGLISH_GUARDRAIL_EXAMPLES]
if len(english_records) != ENGLISH_GUARDRAIL_EXAMPLES:
    raise RuntimeError('Not enough duration-safe English guardrail examples.')

manifest_dir = ROOT / 'artifacts' / 'nemo_container_manifests'
train_manifest = write_nemo_manifest(train_records, manifest_dir, 'nl_train')
validation_manifest = write_nemo_manifest(validation_records, manifest_dir, 'nl_validation')
test_manifest = write_nemo_manifest(test_records, manifest_dir, 'nl_test')
english_manifest = write_nemo_manifest(english_records, manifest_dir, 'en_guardrail')
actual_examples = {
    'train': len(train_records), 'validation': len(validation_records),
    'test': len(test_records),
}
sample_exposures = COMMON_EFFECTIVE_BATCH_SIZE * MAX_STEPS

job_dir = ROOT / 'artifacts' / 'nemo_container_job'
job_dir.mkdir(parents=True, exist_ok=True)
job_config_path = job_dir / 'config.json'
job_config = {
    'root': str(ROOT), 'container_image': NEMO_CONTAINER_IMAGE,
    'model_id': MODEL_ID, 'language_name': LANGUAGE_NAME,
    'language_config': LANGUAGE_CONFIG,
    'train_manifest': str(train_manifest),
    'validation_manifest': str(validation_manifest),
    'test_manifest': str(test_manifest), 'english_manifest': str(english_manifest),
    'requested_examples': {
        'train': TRAIN_EXAMPLES, 'validation': VALIDATION_EXAMPLES, 'test': TEST_EXAMPLES,
    },
    'max_audio_seconds': MAX_AUDIO_SECONDS, 'max_steps': MAX_STEPS,
    'learning_rate': LEARNING_RATE,
    'trainable_encoder_layers': TRAINABLE_ENCODER_LAYERS,
    'train_batch_size': TRAIN_BATCH_SIZE, 'eval_batch_size': EVAL_BATCH_SIZE,
    'accumulate_grad_batches': ACCUMULATE_GRAD_BATCHES,
    'validation_every_examples': VALIDATION_EVERY_EXAMPLES,
    'random_seed': RANDOM_SEED, 'enable_tensorboard': ENABLE_TENSORBOARD,
}
job_config_path.write_text(json.dumps(job_config, indent=2) + '\n', encoding='utf-8')
print({
    'actual_examples': actual_examples, 'sample_exposures': sample_exposures,
    'effective_training_passes': round(sample_exposures / len(train_records), 2),
    'train_minutes': round(total_duration(train_records) / 60, 1),
    'validation_minutes': round(total_duration(validation_records) / 60, 1),
    'test_minutes': round(total_duration(test_records) / 60, 1),
    'container_job_config': str(job_config_path),
})


## 3. Pull the NGC image and run the isolated GPU job

The NGC personal key is accepted through a hidden prompt, used only by a temporary Docker configuration to pull the pinned image, and never passed into the training container or written to the repository. The job mounts only this repository, uses GPU 0, streams logs into the notebook, and runs as your host UID/GID so artifacts remain editable. Set `PULL_IMAGE = False` after the exact image is cached to skip registry login on later runs.

The first pull is large and can take substantial time. Container availability does not prove driver compatibility: the worker fails early if CUDA is not visible.


In [ ]:
if PULL_IMAGE:
    ngc_api_key = getpass('NGC personal API key (hidden; used only to pull the image): ').strip()
    if not ngc_api_key:
        raise ValueError('An NGC personal API key is required to pull the NeMo Speech image.')
    with TemporaryDirectory(prefix='nemo-container-docker-auth-') as docker_config:
        docker_base = ['docker', '--config', docker_config]
        subprocess.run(
            [*docker_base, 'login', 'nvcr.io', '--username', '$oauthtoken', '--password-stdin'],
            input=ngc_api_key + '\n', text=True, check=True,
        )
        del ngc_api_key
        subprocess.run([*docker_base, 'pull', NEMO_CONTAINER_IMAGE], check=True)
else:
    subprocess.run(['docker', 'image', 'inspect', NEMO_CONTAINER_IMAGE], check=True)

cuda_smoke_command = [
    'docker', 'run', '--rm', '--gpus', 'device=0', NEMO_CONTAINER_IMAGE,
    'python', '-c',
    'import nemo, nemo.collections.asr as nemo_asr, torch; assert torch.cuda.is_available(); '
    'print({"nemo": getattr(nemo, "__version__", "unknown"), '
    '"torch": torch.__version__, "cuda": torch.version.cuda, '
    '"gpu": torch.cuda.get_device_name(0), "asr": nemo_asr.__name__})',
]
print({'cuda_smoke_command': cuda_smoke_command})
subprocess.run(cuda_smoke_command, check=True)

container_command = [
    'docker', 'run', '--rm',
    '--gpus', 'device=0', '--ipc=host', '--shm-size=16g',
    '--ulimit', 'memlock=-1', '--ulimit', 'stack=67108864',
    '--user', f'{os.getuid()}:{os.getgid()}',
    '--mount', f'type=bind,source={ROOT},target={ROOT}',
    '--workdir', str(ROOT),
    '--env', 'HOME=/tmp/nemo-user',
    '--env', 'HF_HOME=' + str(ROOT / '.cache' / 'huggingface'),
    '--env', 'TOKENIZERS_PARALLELISM=false',
    '--env', 'PYTHONUNBUFFERED=1',
    '--env', 'PYTHONPATH=' + str(ROOT / 'src'),
    NEMO_CONTAINER_IMAGE,
    'python', '-u', str(ROOT / 'scripts' / 'run_nemo_speech_container_finetune.py'),
    '--config', str(job_config_path),
]
print({'container_command_without_credentials': container_command})
subprocess.run(container_command, check=True)


## 4. Inspect the container result

The container writes `artifacts/parakeet-ctc-0.6b-nl-container.nemo`, container-specific checkpoints, a separate TensorBoard run, and `artifacts/lab2_container_run_summary.json`. Compare the recorded container/NeMo/PyTorch/CUDA versions with standard Lab 2 before interpreting differences as model-quality effects.


In [ ]:
summary_path = ROOT / 'artifacts' / 'lab2_container_run_summary.json'
if not summary_path.is_file():
    raise RuntimeError('The container job did not produce its run summary.')
container_summary = json.loads(summary_path.read_text(encoding='utf-8'))
container_summary


## Interpretation and ease-of-use boundary

This route is easier when dependency reproducibility is the main problem: the container owns its CUDA/PyTorch/NeMo stack and leaves the host environment responsible only for Jupyter, dataset preparation, Docker, and the NVIDIA runtime. It is not automatically lighter or faster: the image pull is large, host-driver compatibility remains a gate, notebook-to-container orchestration adds a mount boundary, and the container release can differ from the pip-pinned standard Lab 2 runtime. Treat the two notebooks as alternative execution paths, not directly comparable experiments unless model, data, controls, decoder, and software versions all match.
